# Exploration des sources de données

Objectif : regarder la structure réelle de chaque dataset (colonnes, langue, taille, exemples) avant d'écrire la logique de conversion dans `scripts/extraction.py`.

Pour ne pas surcharger le notebook, chaque section n'affiche qu'un seul exemple complet. Le reste est sauvegardé dans `data/samples/` via `scripts/sampling.py`, à consulter directement si besoin de voir plus d'exemples.

In [1]:
import sys
sys.path.append("..")

from datasets import load_dataset
from dotenv import load_dotenv
from scripts.sampling import save_sample, first_split

load_dotenv()

/home/rapha/ia-engineer/llm-finetuning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## MediQAl (FR)

MediQAl est un dataset français de question réponse médicale, construit à partir de vrais examens médicaux français (41 spécialités, 32 603 questions au total sur les trois configs). Objectif du dataset : évaluer le rappel de connaissances factuelles et le raisonnement clinique. Licence CC-BY-4.0.

MediQAl propose trois configs : `oeq` (questions ouvertes, réponse rédigée librement), `mcqm` (QCM à réponses multiples) et `mcqu` (QCM à réponse unique).

Pour l'instant je charge seulement `oeq`. Raison : c'est le format le plus proche de ce qu'on veut au final, un agent qui répond en langage libre plutôt qu'un agent qui coche des cases, et FrenchMedMCQA couvre déjà le format QCM côté français. Si le volume `oeq` ne suffit pas pour le SFT, `mcqm` et `mcqu` restent disponibles en complément.

In [2]:
mediqa = load_dataset("ANR-MALADES/MediQAl", "oeq")
mediqa

DatasetDict({
    test: Dataset({
        features: ['id', 'clinical_case', 'cc_question_number', 'question', 'answer', 'medical_subject', 'question_type'],
        num_rows: 4969
    })
})

In [3]:
_, mediqa_data = first_split(mediqa)
print("lignes :", len(mediqa_data))

lignes : 4969


In [4]:
mediqa_data.features

{'id': Value('string'),
 'clinical_case': Value('string'),
 'cc_question_number': Value('string'),
 'question': Value('string'),
 'answer': Value('string'),
 'medical_subject': Value('string'),
 'question_type': Value('string')}

Un exemple complet, pour voir le format réel (les 19 autres sont dans `data/samples/mediqa/`) :

In [5]:
mediqa_data[0]

{'id': '1',
 'clinical_case': 'Homme, 58 ans, majoration de dyspnée chez un BPCO connu depuis 20 ans.\n\n Examen d’entrée : \n Constantes :\n - FC = 130bpm\n - PA = 130/80 aux deux bras\n - FR = 40/min\n - SpO2 = 80%\n - T = 38 ,5°C\n Tirage sus-sternal\n Cyanose\n Sueurs\n Auscultation : \n - Ronchi bilatéraux et sibilants\n - Tympanisme bilatéral\n Hépatomégalie douloureuse\n Reflux hépato-jugulaire\n Turgescence jugulaire\n Hippocratisme digital\n Cyanose unguéale\n \n ATCD \n Fumeur (80 PA)\n BPCO\n \n Traitement habituel\n β2 mimétiques \n \n HDM : depuis 1 semaine : ne peut plus fumer, expectorations sales \n \n 1h après, malgré le traitement initial, patient tient des propos incohérents et désorienté. ',
 'cc_question_number': '1',
 'question': 'Quels sont éléments de gravité au moment de l’arrivée du patient ?',
 'answer': 'FC>125bpm\n Terrain : ATCD de BPCO, insuffisance respiratoire chronique (hippocratisme digital)\n Signes de détresse respiratoire avec hypoxie et hypercapni

Un seul split (`test`), 4969 exemples. Chaque exemple contient un cas clinique, une question posée sur ce cas et une réponse rédigée, avec un sujet médical et un type de question en plus. Bon candidat pour le SFT.

Je vérifie s'il y a des doublons. Premier essai, je compare seulement (question, réponse) :

In [6]:
# Premier essai : comparer seulement (question, réponse)
paires = [(ex["question"], ex["answer"]) for ex in mediqa_data]
print("lignes :", len(paires))
print("uniques (question, réponse) :", len(set(paires)))

from collections import Counter

compteur = Counter(paires)
doublons = [paire for paire, n in compteur.items() if n > 1]
doublons

lignes : 4969
uniques (question, réponse) : 4967


[('Quel diagnostic évoquez-vous ?', 'maladie de Pompe'),
 ('Quel diagnostic faites-vous ?', 'Hypoplasie du cœur gauche')]

2 doublons ressortent, mais rien ne dit que c'est le même exemple. Je regarde le cas clinique derrière chacun :

In [7]:
for question, answer in doublons:
    cas = [ex["clinical_case"] for ex in mediqa_data if ex["question"] == question and ex["answer"] == answer]
    print("question :", question)
    print("cas cliniques distincts :", len(set(cas)))
    for c in cas:
        print(" -", c[:100].replace("\n", " "))
    print()

question : Quel diagnostic évoquez-vous ?
cas cliniques distincts : 2
 - Un nourrisson de 6 mois consulte aux urgences pour une insuffisance cardiaque récente. Il est hypoto
 - Un nourrisson de 8 mois est hospitalisé pour insuffisance cardiaque. Il a une cardiomyopathie hypert

question : Quel diagnostic faites-vous ?
cas cliniques distincts : 2
 - Un couple jeune vous demande votre avis après une échocardiographie fœtal ayant montré une asymétrie
 - Le médecin transporteur du SAMU prend en charge un nouveau-né à terme en salle de travail pour une h



Dans les deux cas, ce sont bien 2 cas cliniques différents qui posent la même question générique (« Quel diagnostic évoquez-vous ? ») avec une réponse courte identique. Pas de vrais doublons donc, mais ça montre qu'il faut comparer le cas clinique complet, pas seulement la question et la réponse :

In [8]:
# Comparaison correcte : cas clinique inclus
contenus = [(ex["clinical_case"], ex["question"], ex["answer"]) for ex in mediqa_data]
print("lignes :", len(contenus))
print("uniques (cas clinique, question, réponse) :", len(set(contenus)))

lignes : 4969
uniques (cas clinique, question, réponse) : 4969


Confirmation : aucun doublon une fois le cas clinique inclus dans la comparaison.

In [9]:
save_sample(mediqa, "mediqa")

mediqa/test : 20 exemples écrits dans /home/rapha/ia-engineer/llm-finetuning/data/samples/mediqa/test.json


## FrenchMedMCQA

Premier dataset français public de QCM médicaux, construit à partir de vraies questions d'examens du diplôme de spécialisation en pharmacie en France. Publié par Labrak et al. en 2022. Le dataset original contient 3105 questions, ce mirroir HF en contient 1080 (595 + 164 + 321 sur les trois splits).

In [10]:
frenchmedmcqa = load_dataset("nthngdy/frenchmedmcqa")
frenchmedmcqa

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer_a', 'answer_b', 'answer_c', 'answer_d', 'answer_e', 'correct_answers', 'number_correct_answers'],
        num_rows: 595
    })
    validation: Dataset({
        features: ['id', 'question', 'answer_a', 'answer_b', 'answer_c', 'answer_d', 'answer_e', 'correct_answers', 'number_correct_answers'],
        num_rows: 164
    })
    test: Dataset({
        features: ['id', 'question', 'answer_a', 'answer_b', 'answer_c', 'answer_d', 'answer_e', 'correct_answers', 'number_correct_answers'],
        num_rows: 321
    })
})

In [11]:
_, frenchmedmcqa_data = first_split(frenchmedmcqa)
print("lignes :", len(frenchmedmcqa_data))

lignes : 595


In [12]:
frenchmedmcqa_data.features

{'id': Value('string'),
 'question': Value('string'),
 'answer_a': Value('string'),
 'answer_b': Value('string'),
 'answer_c': Value('string'),
 'answer_d': Value('string'),
 'answer_e': Value('string'),
 'correct_answers': Value('int64'),
 'number_correct_answers': ClassLabel(names=['1', '2', '3', '4', '5'])}

Un exemple complet (les 19 autres sont dans `data/samples/frenchmedmcqa/`) :

In [13]:
frenchmedmcqa_data[0]

{'id': '230bac49b0fe863b772410bc8d01a025f63c3c999065480131d6334abd2efeff',
 'question': 'Parmi les affirmations suivantes, une seule est fausse, indiquer laquelle: les particules alpha',
 'answer_a': "Sont formées de noyaux d'hélium",
 'answer_b': 'Sont peu pénétrantes',
 'answer_c': "Toute l'énergie qu'elles transportent est cédée au long d'un parcours de quelques centimètres dans l'air",
 'answer_d': 'Sont arrêtées par une feuille de papier',
 'answer_e': 'Sont peu ionisantes',
 'correct_answers': 4,
 'number_correct_answers': 0}

In [14]:
from collections import Counter

feature = frenchmedmcqa_data.features["number_correct_answers"]
Counter(feature.int2str(n) for n in frenchmedmcqa_data["number_correct_answers"])

Counter({'1': 595})

Trois splits (train 595, validation 164, test 321). QCM à 5 propositions. `number_correct_answers` semblait valoir 0 pour tout le monde dans l'aperçu, mais c'est un `ClassLabel` : la valeur brute 0 décode vers le label "1", pas vers "0 bonne réponse". Une fois décodé, les 1080 exemples de ce mirroir HF n'ont chacun qu'une seule bonne réponse. `correct_answers` est donc directement l'index (0 à 4) de la proposition correcte, pas besoin de gérer des réponses multiples. Il faudra reformuler ce QCM en format question/réponse pour le SFT.

In [15]:
for name, split_data in frenchmedmcqa.items():
    print(name, "lignes :", len(split_data), "ids uniques :", len(set(split_data["id"])))

toutes_ids = {name: set(split_data["id"]) for name, split_data in frenchmedmcqa.items()}
print("fuite train/validation :", len(toutes_ids["train"] & toutes_ids["validation"]))
print("fuite train/test :", len(toutes_ids["train"] & toutes_ids["test"]))
print("fuite validation/test :", len(toutes_ids["validation"] & toutes_ids["test"]))

train lignes : 595 ids uniques : 594
validation lignes : 164 ids uniques : 164
test lignes : 321 ids uniques : 321
fuite train/validation : 0
fuite train/test : 0
fuite validation/test : 0


Un doublon exact dans le split train (même id, donc même contenu) : une question sur Streptococcus pneumoniae apparaît deux fois. Pas de fuite entre les splits, aucun id n'est partagé entre train, validation et test.

In [16]:
save_sample(frenchmedmcqa, "frenchmedmcqa")

frenchmedmcqa/train : 20 exemples écrits dans /home/rapha/ia-engineer/llm-finetuning/data/samples/frenchmedmcqa/train.json
frenchmedmcqa/validation : 20 exemples écrits dans /home/rapha/ia-engineer/llm-finetuning/data/samples/frenchmedmcqa/validation.json
frenchmedmcqa/test : 20 exemples écrits dans /home/rapha/ia-engineer/llm-finetuning/data/samples/frenchmedmcqa/test.json


## MedQuAD (EN)

MedQuAD est un dataset anglais construit en collectant des paires question réponse sur 12 sites du NIH (National Institutes of Health) américain, comme cancer.gov ou MedlinePlus. Publié par Ben Abacha et Demner-Fushman en 2019. Le dataset original contient 47 457 paires, ce mirroir HF en contient 16407.

In [17]:
medquad = load_dataset("keivalya/MedQuad-MedicalQnADataset")
medquad

DatasetDict({
    train: Dataset({
        features: ['qtype', 'Question', 'Answer'],
        num_rows: 16407
    })
})

In [18]:
_, medquad_data = first_split(medquad)
print("lignes :", len(medquad_data))

lignes : 16407


In [19]:
medquad_data.features

{'qtype': Value('string'),
 'Question': Value('string'),
 'Answer': Value('string')}

Un exemple complet (les 19 autres sont dans `data/samples/medquad/`) :

In [20]:
medquad_data[0]

{'qtype': 'susceptibility',
 'Question': 'Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?',
 'Answer': 'LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.'}

Un seul split (`train`), 16407 exemples, en anglais. Structure simple : question, réponse et un type de question (`qtype`). À voir si on traduit ou si on filtre selon la langue cible du projet.

In [21]:
contenus = [(ex["Question"], ex["Answer"]) for ex in medquad_data]
print("lignes :", len(contenus))
print("uniques (question, réponse) :", len(set(contenus)))
print("questions uniques :", len(set(medquad_data["Question"])))

lignes : 16407
uniques (question, réponse) : 16359
questions uniques : 14979


48 doublons exacts (question, réponse) sur 16407 lignes, à nettoyer avant l'agrégation. Les questions se répètent aussi plus largement (14979 questions uniques pour 16407 lignes), mais ça vient de la structure de MedQuAD : une même question générique (« What is (are) High Blood Cholesterol ? ») revient plusieurs fois avec des réponses différentes, chacune couvrant une section différente de la fiche source (causes, symptômes, traitement...). Ce n'est pas un doublon, juste plusieurs réponses pour la même question.

In [22]:
save_sample(medquad, "medquad")

medquad/train : 20 exemples écrits dans /home/rapha/ia-engineer/llm-finetuning/data/samples/medquad/train.json


## UltraMedical-Preference (EN, pour le DPO)

UltraMedical-Preference est un dataset anglais de préférences, construit par TsinghuaC3I pour le DPO. Pour chaque prompt médical, plusieurs modèles (GPT-3.5, GPT-4, Llama 3, Qwen1.5, Mixtral...) génèrent une réponse, notée et classée par GPT-4, ce qui donne les paires chosen / rejected. Licence MIT.

In [23]:
ultramedical_preference = load_dataset("TsinghuaC3I/UltraMedical-Preference")
ultramedical_preference

DatasetDict({
    train: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback'],
        num_rows: 109353
    })
    validation: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback'],
        num_rows: 2232
    })
    test: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback'],
        num_rows: 777
    })
})

In [24]:
_, preference_data = first_split(ultramedical_preference)
print("lignes :", len(preference_data))

lignes : 109353


In [25]:
preference_data.features

{'prompt_id': Value('string'),
 'label_type': Value('string'),
 'prompt': Value('string'),
 'chosen': List({'content': Value('string'), 'role': Value('string')}),
 'rejected': List({'content': Value('string'), 'role': Value('string')}),
 'metadata': {'golden_answer': Value('string'),
  'chosen': {'model': Value('string'),
   'score': Value('float64'),
   'rank': Value('int64'),
   'evaluation': Value('string')},
  'rejected': {'model': Value('string'),
   'score': Value('float64'),
   'rank': Value('int64'),
   'evaluation': Value('string')}},
 'feedback': Value('string')}

Un exemple complet (les 19 autres sont dans `data/samples/ultramedical_preference/`) :

In [26]:
preference_data[0]

{'prompt_id': 'WikiInstruct,8304',
 'label_type': 'length',
 'prompt': 'Investigate the intricacies of immunometabolism, a distinct subfield of immunology that examines the interconnection between cellular metabolic processes and the functional attributes of immune cells. Clarify the mechanisms by which this symbiosis modulates the comprehensive immune response, and analyze its integration within the broader spectrum of immunological studies, with an emphasis on the implications for metabolic diseases and the consequential effects on the proficiency of the immune system.',
 'chosen': [{'content': 'Investigate the intricacies of immunometabolism, a distinct subfield of immunology that examines the interconnection between cellular metabolic processes and the functional attributes of immune cells. Clarify the mechanisms by which this symbiosis modulates the comprehensive immune response, and analyze its integration within the broader spectrum of immunological studies, with an emphasis on 

Trois splits (train 109353, validation 2232, test 777). Langue : anglais, prompts et réponses inclus. Chaque exemple a un prompt, une réponse `chosen` et une réponse `rejected`, avec des métadonnées de notation par modèle (score, rang, évaluation). C'est le format attendu pour le DPO.

Point à trancher : le SFT se fait en français (MediQAl, FrenchMedMCQA) alors que ce dataset DPO est entièrement en anglais. Il faudra décider si on traduit ce dataset, si on trouve une alternative en français, ou si on assume un DPO en anglais sur un modèle fine tuné en français.

In [27]:
ids = {name: set(split_data["prompt_id"]) for name, split_data in ultramedical_preference.items()}
for name, split_data in ultramedical_preference.items():
    print(name, "lignes :", len(split_data), "prompt_id uniques :", len(ids[name]))

print("fuite train/validation :", len(ids["train"] & ids["validation"]))
print("fuite train/test :", len(ids["train"] & ids["test"]))
print("fuite validation/test :", len(ids["validation"] & ids["test"]))

train lignes : 109353 prompt_id uniques : 77046
validation lignes : 2232 prompt_id uniques : 2215
test lignes : 777 prompt_id uniques : 776
fuite train/validation : 1209
fuite train/test : 0
fuite validation/test : 0


Chaque split a moins de prompt_id uniques que de lignes (par exemple 77046 prompt_id pour 109353 lignes en train), ce qui est normal pour du DPO : un même prompt peut avoir plusieurs paires chosen/rejected.

Le point important : 1209 prompt_id apparaissent à la fois dans train et validation, soit environ 55% de la validation (2215 prompt_id uniques). Aucune fuite entre train/test ni validation/test. Pour le DPO, il faudra retirer ces prompts partagés du split train ou du split validation avant l'entraînement, sinon la validation n'est plus indépendante de l'entraînement.

In [28]:
save_sample(ultramedical_preference, "ultramedical_preference")

ultramedical_preference/train : 20 exemples écrits dans /home/rapha/ia-engineer/llm-finetuning/data/samples/ultramedical_preference/train.json
ultramedical_preference/validation : 20 exemples écrits dans /home/rapha/ia-engineer/llm-finetuning/data/samples/ultramedical_preference/validation.json
ultramedical_preference/test : 20 exemples écrits dans /home/rapha/ia-engineer/llm-finetuning/data/samples/ultramedical_preference/test.json


## Synthèse

Quatre datasets explorés, à combiner en un schéma commun avant d'écrire la logique dans `scripts/extraction.py`.

Pour le SFT (français) :
- MediQAl (`oeq`) : 4969 exemples, cas clinique + question + réponse rédigée, split `test` uniquement
- FrenchMedMCQA : 595 / 164 / 321 (train / validation / test), QCM à 5 propositions, à reformuler en question/réponse
- MedQuAD : 16407 exemples en anglais, split `train` uniquement

Pour le DPO :
- UltraMedical-Preference : 109353 / 2232 / 777 (train / validation / test), en anglais, format prompt / chosen / rejected déjà prêt

Points à trancher avant l'extraction :
- Langue : le SFT visé est en français mais MedQuAD et UltraMedical-Preference sont en anglais. Il faut décider entre traduction, filtrage, ou DPO en anglais sur un modèle fine tuné en français.
- MediQAl et MedQuAD n'ont pas de split train/test, il faudra en créer un nous mêmes pour l'évaluation.

Doublons et fuites entre splits, vérifiés source par source :
- MediQAl : aucun doublon (une fois le cas clinique complet pris en compte, pas juste question/réponse).
- FrenchMedMCQA : un doublon exact dans train, aucune fuite entre splits.
- MedQuAD : 48 doublons exacts (question, réponse) sur 16407 lignes, à nettoyer avant l'agrégation.
- UltraMedical-Preference : aucune fuite train/test ni validation/test, mais 1209 prompt_id partagés entre train et validation (environ 55% de la validation), à corriger avant d'entraîner le DPO.

Des échantillons de chaque split sont sauvegardés dans `data/samples/` via `scripts/sampling.py`, pour relire la structure sans recharger les datasets.